In [1]:
import pandas as pd
import numpy as np
import gc
import time

data = pd.read_pickle('../data/merged_clean.pkl')
label_col = "Label"

y = data[label_col].copy()

constant_cols = [c for c in data.columns if c != label_col and data[c].nunique() <= 1]
drop_cols = constant_cols + [label_col, 'Destination Port', 'Fwd Header Length.1']

# Build X directly at float32 to avoid a full float64 copy
keep = [c for c in data.columns if c not in drop_cols]
X = data[keep].astype(np.float32)

del data
gc.collect()

y = y.replace({
    'Web Attack - Brute Force': 'Web Attack',
    'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
})
mask = ~y.isin(['Heartbleed', 'Infiltration'])
X, y = X[mask], y[mask]
gc.collect()

print("Shape:", X.shape, " Classes:", y.nunique())


Shape: (2520751, 68)  Classes: 11


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)
print("Train:", X_train.shape, " Test:", X_test.shape)

# Free the full copies - we only need the training fold for tuning
del X, y
gc.collect()

Train: (1764525, 68)  Test: (756226, 68)


10

In [3]:
# Tuning on the full 1.76M rows is not feasible on this hardware:
# a single DT fit takes ~340s, so 10-fold CV across a parameter grid
# would run to many hours. Parameters are selected on a stratified
# 400k subsample, then the final models are refitted on the full
# training set in notebook 05.
X_tune, _, y_tune, _ = train_test_split(
    X_train, y_train, train_size=400_000, stratify=y_train, random_state=42
)
print(y_tune.value_counts())
gc.collect()

Label
BENIGN              332449
DoS Hulk             27428
DDoS                 20314
PortScan             14392
DoS GoldenEye         1632
FTP-Patator            941
DoS slowloris          854
DoS Slowhttptest       830
SSH-Patator            511
Web Attack             340
Bot                    309
Name: count, dtype: int64


0

In [4]:
from imblearn.pipeline import Pipeline          # imblearn's, NOT sklearn's
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier
from collections import Counter

# Scale the SMOTE target to the subsample: 50k on 1.76M rows is
# ~2.8%, so the equivalent on 400k is ~11k.
TUNE_TARGET = 11_000
counts = Counter(y_tune)
strategy = {c: TUNE_TARGET for c, n in counts.items() if n < TUNE_TARGET}
print("Oversampling:", {k: f"{counts[k]}->{v}" for k, v in strategy.items()})

dt_pipe = Pipeline([
    ('smote', SMOTE(sampling_strategy=strategy, random_state=42, k_neighbors=5)),
    ('clf', DecisionTreeClassifier(random_state=42)),
])

Oversampling: {'DoS Slowhttptest': '830->11000', 'DoS slowloris': '854->11000', 'DoS GoldenEye': '1632->11000', 'FTP-Patator': '941->11000', 'SSH-Patator': '511->11000', 'Web Attack': '340->11000', 'Bot': '309->11000'}


In [5]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
import time

dt_grid = {
    'clf__max_depth': [10, 20, 30],
    'clf__min_samples_split': [2, 10],
    'clf__min_samples_leaf': [1, 5],
}
# 4 x 3 x 2 = 24 combinations x 10 folds = 240 fits

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

t = time.time()
dt_search = GridSearchCV(
    dt_pipe, dt_grid, scoring='f1_macro', cv=cv,
    n_jobs=1, verbose=2          # n_jobs=1 to protect your 8 GB
)
dt_search.fit(X_tune, y_tune)
print(f"\nCompleted in {(time.time()-t)/60:.1f} min")
print("Best params:", dt_search.best_params_)
print("Best CV macro F1:", dt_search.best_score_)

Fitting 10 folds for each of 12 candidates, totalling 120 fits
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  44.4s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  29.2s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  25.1s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  24.9s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  29.2s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  23.3s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  23.3s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  27.8s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, clf__min_samples_split=2; total time=  33.8s
[CV] END clf__max_depth=10, clf__min_samples_leaf=1, cl